# 14 — LLM (Claude) vs the production model on the held-out set

**Question.** How does a Claude few-shot classifier compare to the trained production model on the
held-out set (issues 105-114), especially on the editorial **triangle** (`policy_practice_research`,
`political_environment_key_organisations`, `what_matters_ed`)?

**Hypothesis (being tested, not assumed).** We expected the LLM, *following the curator's written
boundary rules in the prompt*, to do **better** on the fuzzy triangle than a model that must learn the
boundary from data. This notebook tests that — and the first run **refuted** it (Claude well below the
trained model). So we treat it carefully.

**Fair-comparison controls.**
1. Few-shot construction is aligned **exactly** to NB06 (the run that scored ~0.717): 2 random examples
   per class, grouped under category headers. (An earlier flat-list version depressed the score.)
2. **Val-set control (Step 5):** the same classifier is also run on the val set. If it reproduces
   NB06's ~0.717 on val but scores low on held-out, the held-out is genuinely Claude's blind spot
   (a *data* effect). If val is also low, the gap is *setup/model*, not data.

**Self-contained.** Builds the held-out set, runs the production model (corrected title+description
input), runs Claude on held-out and on val — no dependency on NB11.

**Model.** `claude-sonnet-4-6` (current alias; NB06 used `claude-sonnet-4-20250514`, deprecated). Pin
via the alias — never a date suffix. The val control isolates any model-version effect.

In [1]:
import sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer
from anthropic import Anthropic
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / '.env')

MODEL = 'claude-sonnet-4-6'   # current alias; do NOT append a date suffix
client = Anthropic()          # reads ANTHROPIC_API_KEY from env / .env

MODEL_DIR = ROOT / 'models' / 'runs' / 'v1_2026-05-16'
with open(MODEL_DIR / 'run_metadata.json') as f:
    run_meta = json.load(f)

CONCRETE  = ['edtech', 'four_nations', 'teacher_rrd']
EDITORIAL = ['policy_practice_research', 'political_environment_key_organisations', 'what_matters_ed']
CONTENT_CLASSES = CONCRETE + EDITORIAL
print('model:', MODEL)

model: claude-sonnet-4-6


## Step 1 — Build the held-out set (issues 105-114), same pipeline as training

In [2]:
from src.training_data import s01_clean as s01
from src.training_data import s02_preprocess as s02
from src.training_data.s03_split import VALID_THEMES

RAW_CSV = ROOT / 'data' / 'interim' / 'extracted_newsletters.csv'
HELD_OUT_ISSUES = range(105, 115)

df = pd.read_csv(RAW_CSV)
df['newsletter_number'] = pd.to_numeric(df['newsletter_number'], errors='coerce')
df = df[df['newsletter_number'].isin(HELD_OUT_ISSUES)].copy()

obj_cols = [c for c in df.columns if df[c].dtype == object]
for c in obj_cols:
    df[c] = s01.clean_series(df[c])
df = s01.fix_artifacts(df, obj_cols)
df = s01.drop_missing_essentials(df)
df = s01.deduplicate(df)
df = s02.add_themes(df)
df = s02.add_domain_and_org(df)
df['item_type'] = df.apply(s02.classify_item_type, axis=1)
df = s02.add_text_features(df)

test_df = df[df['new_theme'].isin(CONTENT_CLASSES)].copy().reset_index(drop=True)

# Model input = title + description (text_clean/text), matching training. NOT title-only.
def model_input(row):
    for c in ('text_clean', 'text'):
        if c in row and pd.notna(row[c]) and str(row[c]).strip():
            return str(row[c])
    title = str(row['title']) if pd.notna(row.get('title')) else ''
    desc  = str(row['description']) if pd.notna(row.get('description')) else ''
    return (title + ' ' + desc).strip()

test_df['model_text'] = test_df.apply(model_input, axis=1)
y_true = test_df['new_theme'].values
print(f'Held-out test items (6 content classes): {len(test_df)}')
print(test_df['new_theme'].value_counts().to_string())

  Dropped 21 rows with missing title/link or URL-as-title. Remaining: 136
  After title+link dedup: 135 rows (removed 1)
  After title-only dedup: 135 rows (removed 0)
  After link-only dedup:  134 rows (removed 1)
  Dropped 0 unsubscribe row(s)
  Unmapped domains: 36 rows
Held-out test items (6 content classes): 116
new_theme
what_matters_ed                            25
political_environment_key_organisations    22
policy_practice_research                   20
teacher_rrd                                18
four_nations                               16
edtech                                     15


/workspaces/AM2_erp_programme_automataion/src/training_data/s02_preprocess.py:750: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  kw_four_nations = theme_norm.str.contains(r"\b(4|four) nations\b", regex=True, na=False)


## Step 2 — Production model (SBERT + LogReg) on the held-out set

In [3]:
classifier = joblib.load(MODEL_DIR / 'classifier.joblib')
encoder = SentenceTransformer(run_meta['embedding_model'])

emb = encoder.encode(test_df['model_text'].tolist(), show_progress_bar=True)
y_pred_prod = classifier.predict(emb)

prod_macro = f1_score(y_true, y_pred_prod, average='macro', zero_division=0)
print(f'Production model — held-out macro-F1: {prod_macro:.3f}  acc: {accuracy_score(y_true, y_pred_prod):.3f}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Production model — held-out macro-F1: 0.725  acc: 0.707


## Step 3 — Claude few-shot classifier (curator boundary rules in the prompt)
Category descriptions reused from NB06 — they encode the curator's editorial boundaries, especially
the triangle ("policy = the research itself; political = the organisation's action; what_matters = the topic").

In [4]:
CATEGORY_DESCRIPTIONS = {
    'edtech': 'Articles about educational technology, AI in education, digital tools for learning, online learning platforms, and technology policy in schools.',
    'four_nations': 'Articles specifically about education in Scotland, Wales, Northern Ireland, or devolved education policy. Must have a clear geographic focus on one of the four nations.',
    'policy_practice_research': 'Research reports, academic studies, evidence reviews, and practice-focused publications about education. The emphasis is on the research or evidence itself, not on the organisation publishing it.',
    'political_environment_key_organisations': "News and announcements from key political and policy organisations (DfE, Ofsted, parliamentary committees, think tanks). The emphasis is on the organisation's action or announcement, not on the topic.",
    'teacher_rrd': 'Articles specifically about teacher recruitment, retention, development, training, pay, workload, or the teaching profession. Must be primarily about teachers as a workforce.',
    'what_matters_ed': 'Articles about broader education issues that matter to children and families: SEND, attendance, mental health, disadvantage, pupil welfare, school meals, poverty.',
}

SYSTEM_PROMPT = """You are a classifier for an education policy newsletter.
Classify each article into exactly one of these categories:

{categories}

Respond with ONLY a JSON object in this format:
{{
  "category": "the_category_name",
  "confidence": 0.0 to 1.0,
  "second_category": "the_second_most_likely_category",
  "second_confidence": 0.0 to 1.0
}}

Do not include any other text."""

def format_system_prompt(category_descriptions, examples=None):
    categories = "\n".join(f"- {n}: {d}" for n, d in category_descriptions.items())
    prompt = SYSTEM_PROMPT.format(categories=categories)
    if examples:
        prompt += "\n\nHere are example articles for each category:\n" + examples
    return prompt

def find_train():
    for p in [ROOT/'data'/'modelling'/'train.csv', ROOT/'data'/'processed'/'train.csv']:
        if p.exists():
            return pd.read_csv(p)
    raise FileNotFoundError('train.csv')
train_df = find_train()
tcol = 'target' if 'target' in train_df.columns else 'new_theme'
xcol = 'text_clean' if 'text_clean' in train_df.columns else 'text'

# Few-shot examples — EXACTLY NB06's construction (the run that scored ~0.717):
# 2 RANDOM examples per class (random_state=42), grouped UNDER each category header so Claude
# knows which example belongs to which class. (NB14 originally used a flat "[cls] text" list +
# .head(2); that mismatch is the likely cause of the depressed 0.473 — fixed here for a fair test.)
examples_text = ""
for cls in CATEGORY_DESCRIPTIONS:
    cat_df = train_df[train_df[tcol] == cls]
    samples = cat_df.sample(min(2, len(cat_df)), random_state=42)
    examples_text += f"\n{cls}:\n"
    for _, r in samples.iterrows():
        examples_text += f"  - {str(r[xcol])[:200]}\n"
system_prompt_few = format_system_prompt(CATEGORY_DESCRIPTIONS, examples=examples_text)
print('Few-shot prompt built (NB06-style: category-grouped, 2 random per class).')

def classify_article(text, system_prompt, model=MODEL):
    resp = client.messages.create(
        model=model, max_tokens=150, system=system_prompt,
        messages=[{'role': 'user', 'content': f'Classify this article:\n\n{text}'}],
    )
    try:
        return json.loads(resp.content[0].text)
    except (json.JSONDecodeError, IndexError):
        return {'category': 'parse_error'}

Few-shot prompt built (NB06-style: category-grouped, 2 random per class).


In [5]:
import re
# Robust JSON extraction. The plain json.loads(resp.content[0].text) was failing on most responses
# (code fences / preamble / truncation) -> dozens of parse_errors -> a fake 0.149. This version:
#  - joins ALL text blocks, skipping any non-text (e.g. thinking) blocks,
#  - regex-extracts the first {...} (survives ```json fences and stray prose),
#  - bumps max_tokens to 256 so the JSON isn't truncated,
#  - keeps the raw text on failure so any remaining misses can be inspected.
def classify_article(text, system_prompt, model=MODEL):
    resp = client.messages.create(
        model=model, max_tokens=256, system=system_prompt,
        messages=[{'role': 'user', 'content': f'Classify this article:\n\n{text}'}],
    )
    raw = ''.join(getattr(b, 'text', '') for b in resp.content if getattr(b, 'type', None) == 'text')
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            pass
    return {'category': 'parse_error', '_raw': raw[:200]}

print('Robust classify_article installed (regex JSON extraction, max_tokens=256).')

Robust classify_article installed (regex JSON extraction, max_tokens=256).


In [6]:
# Run Claude on every held-out item. ~116 calls — a minute or two, a few cents.
llm_pred, llm_second = [], []
for i, text in enumerate(test_df['model_text']):
    try:
        r = classify_article(text, system_prompt_few)
    except Exception as e:
        r = {'category': 'error', 'second_category': None}
        print(f'  item {i} error: {type(e).__name__}: {e}')
    llm_pred.append(r.get('category', 'parse_error'))
    llm_second.append(r.get('second_category'))
    if (i + 1) % 20 == 0:
        print(f'  {i + 1}/{len(test_df)}')
test_df['llm_pred'] = llm_pred
test_df['llm_second'] = llm_second
y_pred_llm = np.array(llm_pred)
bad = set(y_pred_llm) - set(CONTENT_CLASSES)
print(f'Done. Off-label / failed predictions: {bad or "none"}')

  20/116
  40/116
  60/116
  80/116
  100/116
Done. Off-label / failed predictions: none


## Step 4 — Head-to-head

In [7]:
llm_macro = f1_score(y_true, y_pred_llm, average='macro', zero_division=0)
print('=== Held-out macro-F1 ===')
print(f'  Production (SBERT+LogReg): {prod_macro:.3f}')
print(f'  Claude few-shot:          {llm_macro:.3f}')
print(f'  Delta (Claude - prod):    {llm_macro - prod_macro:+.3f}')

# Top-2 accuracy for Claude (uses its second_category).
llm_top2 = np.mean([yt == p or yt == s for yt, p, s in zip(y_true, y_pred_llm, test_df['llm_second'])])
print(f'\n  Claude top-2 accuracy:    {llm_top2:.3f}')

print('\n=== Per-class F1 (production vs Claude) ===')
rep_p = classification_report(y_true, y_pred_prod, labels=CONTENT_CLASSES, output_dict=True, zero_division=0)
rep_l = classification_report(y_true, y_pred_llm,  labels=CONTENT_CLASSES, output_dict=True, zero_division=0)
cmp = pd.DataFrame({
    'group':   ['concrete' if c in CONCRETE else 'editorial' for c in CONTENT_CLASSES],
    'n':       [int(rep_p[c]['support']) for c in CONTENT_CLASSES],
    'prod_f1': [round(rep_p[c]['f1-score'], 2) for c in CONTENT_CLASSES],
    'llm_f1':  [round(rep_l[c]['f1-score'], 2) for c in CONTENT_CLASSES],
}, index=CONTENT_CLASSES)
cmp['winner'] = np.where(cmp.llm_f1 > cmp.prod_f1, 'Claude', np.where(cmp.prod_f1 > cmp.llm_f1, 'prod', 'tie'))
print(cmp.to_string())
print(f"\n  Editorial-triangle mean F1 — prod {cmp[cmp.group=='editorial'].prod_f1.mean():.2f}  "
      f"Claude {cmp[cmp.group=='editorial'].llm_f1.mean():.2f}")

=== Held-out macro-F1 ===
  Production (SBERT+LogReg): 0.725
  Claude few-shot:          0.634
  Delta (Claude - prod):    -0.091

  Claude top-2 accuracy:    0.879

=== Per-class F1 (production vs Claude) ===
                                             group   n  prod_f1  llm_f1  winner
edtech                                    concrete  15     0.82    0.79    prod
four_nations                              concrete  16     0.90    1.00  Claude
teacher_rrd                               concrete  18     0.89    0.86    prod
policy_practice_research                 editorial  20     0.59    0.24    prod
political_environment_key_organisations  editorial  22     0.50    0.45    prod
what_matters_ed                          editorial  25     0.65    0.47    prod

  Editorial-triangle mean F1 — prod 0.58  Claude 0.39


In [8]:
# Where does Claude rescue the triangle? Items in the editorial classes that prod got wrong but Claude got right.
test_df['_true'] = y_true
test_df['_prod'] = y_pred_prod
tri = test_df[test_df['_true'].isin(EDITORIAL)]
rescued = tri[(tri['_prod'] != tri['_true']) & (tri['llm_pred'] == tri['_true'])]
lost    = tri[(tri['_prod'] == tri['_true']) & (tri['llm_pred'] != tri['_true'])]
print(f'Triangle items Claude got right that production got wrong: {len(rescued)}')
for _, r in rescued.head(8).iterrows():
    print(f"  [{r['_true']}] prod->{r['_prod']:40s} | {str(r.get('title') or '')[:70]}")
print(f'\nTriangle items production got right that Claude got wrong: {len(lost)}')
for _, r in lost.head(8).iterrows():
    print(f"  [{r['_true']}] claude->{r['llm_pred']:40s} | {str(r.get('title') or '')[:70]}")

Triangle items Claude got right that production got wrong: 6
  [political_environment_key_organisations] prod->teacher_rrd                              | Lift Schools - CEO to co-chair new government KS3 Alliance
  [policy_practice_research] prod->edtech                                   | Forum - The Education Endowment Foundation Toolkit: A house built on s
  [what_matters_ed] prod->political_environment_key_organisations  | Schools Week - 'Experts at hand' cash must not plug 'existing gaps', c
  [political_environment_key_organisations] prod->policy_practice_research                 | The Young Foundation - Stop asking communities what they think – and s
  [policy_practice_research] prod->edtech                                   | The Conversation - Dolls beat screens for building children's social s
  [what_matters_ed] prod->political_environment_key_organisations  | The Children's Commissioner - The Big Future

Triangle items production got right that Claude got wrong: 17
  [polic

## Step 5 — Val-set control: does the aligned setup reproduce NB06's ~0.717?

The same Claude few-shot classifier, run on the **val set** (the set NB06 scored ~0.717 on). This
isolates *setup/model* from *data*:
- **val ≈ 0.717, held-out lower** → the held-out is genuinely Claude's blind spot (a *data* effect, the finding stands).
- **val also low** → the gap is the setup/model (e.g. `claude-sonnet-4-6` vs the old `claude-sonnet-4-20250514`), not the data — reconcile before concluding.

In [9]:
# Run the SAME Claude few-shot classifier on the val set (NB06's ~0.717 set). ~167 calls.
val_path = next(p for p in [ROOT/'data'/'modelling'/'val.csv', ROOT/'data'/'processed'/'val.csv'] if p.exists())
val_df = pd.read_csv(val_path)
vtcol = 'target' if 'target' in val_df.columns else 'new_theme'
vxcol = 'text_clean' if 'text_clean' in val_df.columns else 'text'
val_df = val_df[val_df[vtcol].isin(CONTENT_CLASSES)].copy().reset_index(drop=True)

val_pred = []
for i, text in enumerate(val_df[vxcol].fillna('').astype(str)):
    try:
        r = classify_article(text, system_prompt_few)
    except Exception as e:
        r = {'category': 'error'}
    val_pred.append(r.get('category', 'parse_error'))
    if (i + 1) % 40 == 0:
        print(f'  val {i + 1}/{len(val_df)}')

val_macro = f1_score(val_df[vtcol].values, np.array(val_pred), average='macro',
                     zero_division=0, labels=CONTENT_CLASSES)
n_bad_val = int((~pd.Series(val_pred).isin(CONTENT_CLASSES)).sum())
print('\n=== Claude few-shot — reproduction check ===')
print(f'  VAL macro-F1:      {val_macro:.3f}   (NB06 reported ~0.717)   parse/off-label: {n_bad_val})')
print(f'  HELD-OUT macro-F1: {llm_macro:.3f}')
print(f'  Production held-out: {prod_macro:.3f}')
print('\nRead: val≈0.717 & held-out lower -> held-out is Claude\'s blind spot (data, finding stands).')
print('      val also low            -> setup/model gap, not data (reconcile before concluding).')

  val 40/167
  val 80/167
  val 120/167
  val 160/167

=== Claude few-shot — reproduction check ===
  VAL macro-F1:      0.724   (NB06 reported ~0.717)   parse/off-label: 1)
  HELD-OUT macro-F1: 0.634
  Production held-out: 0.725

Read: val≈0.717 & held-out lower -> held-out is Claude's blind spot (data, finding stands).
      val also low            -> setup/model gap, not data (reconcile before concluding).


## How to read this (analysis notes, not the write-up)

- **Read Step 5 (val control) FIRST.** It decides what the held-out result means:
  - val ≈ 0.717 → the aligned setup is sound, so the held-out gap is a **data** effect (Claude's blind spot). The refutation of our hypothesis stands.
  - val also low → the gap is **setup/model** (e.g. the model version), not data — reconcile before drawing any conclusion.
- **Headline:** held-out macro-F1 delta + the editorial-triangle mean F1. First (mis-aligned) run gave
  Claude 0.473 vs prod 0.725, with the triangle *worse* for Claude (0.34 vs 0.58) — the **opposite** of
  the hypothesis. Re-check after the few-shot alignment.
- **`rescued` vs `lost`** shows the mechanism: Claude systematically calls research-sector orgs (UKRI,
  British Academy, UPEN) `political` where the curator says `policy` — a *normative divergence* (Claude
  follows the literal rule; the curator's real boundary is idiosyncratic and learned-from-labels).
- **Caveats:** small n (per-class F1 noisy); cost/latency per item far higher for the LLM; consider the
  hybrid (trained model for concrete classes, LLM only where useful). See
  `docs/decisions/embeddings_and_llm_post_model.md` (which will be corrected once this is settled).